# Lab: Building a Chatbot with Gradio

**Aim:** Build an interactive chatbot interface using Gradio that wraps a conversational language model.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import gradio as gr
from transformers import pipeline, Conversation
print(f'Gradio version: {gr.__version__}')

Gradio version: 6.32.0


## Approach 1: Simple Rule-Based Chatbot with Gradio

In [2]:
# A fallback rule-based bot to show the Gradio interface pattern
# without needing a GPU or large model download.

import random

RESPONSES = {
    'hello': ['Hi there!', 'Hey! How can I help?', 'Hello! Nice to meet you.'],
    'how are you': ['I am doing well, thanks for asking!', 'Pretty good, ready to help.'],
    'what is your name': ['I am a simple AI assistant built in this lab.', 'You can call me LabBot.'],
    'bye': ['Goodbye! See you around.', 'Bye! Have a great day.', 'Take care!'],
    'help': ['I can answer basic questions. Try asking about AI or machine learning topics.'],
    'what is machine learning': [
        'Machine learning is a branch of AI where models learn patterns from data rather than following explicit rules.'
    ],
    'what is deep learning': [
        'Deep learning uses multi-layer neural networks to learn representations of data automatically.'
    ],
    'what is a transformer': [
        'A transformer is a neural network architecture based on self-attention, introduced in the paper Attention Is All You Need (2017).'
    ],
}

def rule_based_respond(message, history):
    msg = message.lower().strip()
    for key, replies in RESPONSES.items():
        if key in msg:
            return random.choice(replies)
    return "I am not sure about that. Try asking something about AI or machine learning."

print('Rule-based bot function defined.')
print('Testing a few responses:')
for test in ['hello', 'what is deep learning', 'bye']:
    print(f'  User: {test}')
    print(f'  Bot : {rule_based_respond(test, [])}')

Rule-based bot function defined.
Testing a few responses:
  User: hello
  Bot : Hello! Nice to meet you.
  User: what is deep learning
  Bot : Deep learning uses multi-layer neural networks to learn representations of data automatically.
  User: bye
  Bot : Goodbye! See you around.


## Approach 2: Connecting to a Real LLM Pipeline

In [3]:
# This shows how to connect a Hugging Face conversational pipeline.
# We demonstrate the pattern here (requires downloading the model weights).

from transformers import AutoTokenizer, AutoModelForCausalLM

def build_llm_chatbot():
    """
    Returns a respond function backed by a small LLM.
    Replace 'microsoft/DialoGPT-small' with any conversational model.
    """
    tokenizer = AutoTokenizer.from_pretrained('microsoft/DialoGPT-small')
    model = AutoModelForCausalLM.from_pretrained('microsoft/DialoGPT-small')
    tokenizer.pad_token = tokenizer.eos_token

    conversation_history = []

    def respond(message, history):
        import torch
        # Encode new user message
        new_input_ids = tokenizer.encode(message + tokenizer.eos_token, return_tensors='pt')
        # Keep only last 3 turns to stay within token limit
        conversation_history.append(new_input_ids)
        if len(conversation_history) > 6:
            conversation_history.pop(0)
        bot_input = torch.cat(conversation_history, dim=-1)
        output = model.generate(bot_input, max_length=200, pad_token_id=tokenizer.eos_token_id,
                                do_sample=True, temperature=0.7, top_p=0.9)
        reply_ids = output[:, bot_input.shape[-1]:]
        reply = tokenizer.decode(reply_ids[0], skip_special_tokens=True)
        conversation_history.append(reply_ids)
        return reply

    return respond

print('LLM chatbot builder function defined.')
print('Note: running build_llm_chatbot() downloads ~350 MB model weights.')

LLM chatbot builder function defined.
Note: running build_llm_chatbot() downloads ~350 MB model weights.


## Gradio Interface — Rule-Based Bot

In [4]:
# Build and launch the Gradio chat interface.
# share=False for local only; share=True for a public link.

iface = gr.ChatInterface(
    fn=rule_based_respond,
    title='LabBot - AI Chatbot',
    description='A simple chatbot for the Gradio lab. Ask about AI, machine learning, or just say hello.',
    examples=[
        'Hello',
        'What is machine learning?',
        'What is a transformer?',
        'What is your name?',
    ],
    theme=gr.themes.Soft(),
)

print('Gradio interface created.')
print('To launch: iface.launch(share=False)')
print('To get a public URL: iface.launch(share=True)')
print()
print('Sample conversation flow:')
conversation = [
    'Hello',
    'What is machine learning?',
    'What is deep learning?',
    'What is a transformer?',
    'Bye'
]
history = []
for msg in conversation:
    reply = rule_based_respond(msg, history)
    history.append([msg, reply])
    print(f'  User : {msg}')
    print(f'  Bot  : {reply}')

Gradio interface created.
To launch: iface.launch(share=False)
To get a public URL: iface.launch(share=True)

Sample conversation flow:
  User : Hello
  Bot  : Hello! Nice to meet you.
  User : What is machine learning?
  Bot  : Machine learning is a branch of AI where models learn patterns from data rather than following explicit rules.
  User : What is deep learning?
  Bot  : Deep learning uses multi-layer neural networks to learn representations of data automatically.
  User : What is a transformer?
  Bot  : A transformer is a neural network architecture based on self-attention, introduced in the paper Attention Is All You Need (2017).
  User : Bye
  Bot  : Goodbye! See you around.


## Adding System Prompt and Context

In [5]:
SYSTEM_PROMPT = """
You are a helpful AI teaching assistant for a generative AI course.
Keep your answers concise, clear and accurate.
If a student asks something outside your knowledge, say so honestly.
"""

def context_chatbot(message, history):
    # Build a simple context-aware prompt
    context = SYSTEM_PROMPT
    for user_msg, bot_msg in history[-3:]:
        context += f'Student: {user_msg}\nAssistant: {bot_msg}\n'
    context += f'Student: {message}\nAssistant:'

    # With a real LLM you would call model.generate(context)
    # Here we fall back to rules
    return rule_based_respond(message, history)

print('Context-aware chatbot defined.')
print('System prompt characters:', len(SYSTEM_PROMPT))

Context-aware chatbot defined.
System prompt characters: 185


## Observations

- Gradio's `ChatInterface` takes a simple `fn(message, history) -> reply` function and builds a full chat UI automatically.
- The `examples` list shows clickable example prompts on the interface, which is useful for demos.
- For production chatbots, the respond function would call a real LLM (OpenAI API, Hugging Face Inference API, or a locally deployed model) instead of rule-based logic.
- Conversation history is passed as a list of [user, bot] pairs, so the function can maintain context across turns.
- `share=True` on launch creates a public tunnel URL valid for 72 hours, useful for sharing demos without deploying a server.